<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Whisper sizes on a laptop

**[Whisper sizes on a laptop](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/)** · Speech AI: Transcribe, Understand and Speak · Module 4, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score the recorded transcripts of three Whisper sizes and Vosk on the same test calls, by slice, put the word error rate next to the speed, and run Whisper `tiny` yourself to see its speed on your computer.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 80 MB with the Whisper model). No account, no key and no graphics card. |
| **You should know** | WER and the normalisation guide, from [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/) and [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M04-L02-whisper-sizes-on-a-laptop/whisper-sizes-on-a-laptop-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 4 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/comparison_table.json": "1cc9542aa8c3b930019b4209d9ebbcf3844f77d5a17dc0417fc02617a114d7b0",
    "data/recordings/asr/vosk-small.jsonl": "cfb072991ed3c2de1a9604b1f001952de061a4231999b4051f2b20f93b7641be",
    "data/recordings/asr/whisper-tiny.jsonl": "d6c12c960a2bc99981f866d674830b7501c01f754ae0f11d8bacf1139aa7285b",
    "data/recordings/asr/whisper-base.jsonl": "2af177c7cef6bcb9a24dabdcb4c108a785d7cf8effae475de053390ec5465c90",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "data/calls/phone/C-0225.wav": "0eab5a465ce8234b1c6e8354f7447cdaa5434eef6d7de6ccfb09ad0978d699b0",
    "data/calls/phone/C-0121.wav": "f9a119e345ca6ecb268ca7289e3c6655662bb0cdfb389161c7054c6b026f315c",
    "data/calls/phone/C-0083.wav": "8896c2f5abf5d1fc71248906bfd6341e778efc4bf6acf24cd0115e000ef90e24",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded transcripts

Tomás says: "Use the biggest Whisper. Speech recognition is solved." Priya wants numbers first. Running four recognisers on every call takes time, so the course team ran them once on the same laptop (Apple M4, 4 threads) and recorded every transcript with its compute time:

- **Vosk small**: the older pipeline (acoustic model, pronunciation dictionary and language model), about 40 MB a language. Apache-2.0.
- **Whisper tiny, base and small**: the same encoder–decoder model in three sizes (39, 74 and 244 million parameters), through faster-whisper with 8-bit weights (int8). MIT.

Run the next cell. It loads the labels and the recorded transcripts.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]
methods = ['vosk-small', 'whisper-tiny', 'whisper-base', 'whisper-small']
recorded = {}   # recorded[method][(call_id, kind)] = {"text": ..., "seconds": compute time}
for m in methods:
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{m}.jsonl", encoding="utf-8") if line.strip()]
    recorded[m] = {(r["call_id"], r["kind"]): r for r in rows}
print(len(test), "test calls;", {m: len(v) for m, v in recorded.items()})

> **Check.** 60 test calls; each method has a row for the phone version and the wideband version of each call.

## 2. One score for each method

The next cell defines `score`: WER and CER after the normalisation guide, with the errors summed over all the calls and then divided (`cp.corpus_scores`), and the **real-time factor** (compute time / audio time).

In [ ]:
def score(method, kind="phone", by=None):
    """WER and CER after the guide (errors summed over the calls), and the real-time factor, by slice."""
    groups = {}
    for c in test:
        row = recorded[method].get((c["call_id"], kind))
        if row is None:
            continue
        key = c[by] if by else "all"
        groups.setdefault(key, []).append((c, row))
    out = {}
    for key, items in sorted(groups.items()):
        s = cp.corpus_scores([(reference(c), r["text"], c["language"]) for c, r in items])
        rtf = sum(r["seconds"] for _, r in items) / sum(c["duration"] for c, _ in items)
        out[key] = {"calls": len(items), "wer": round(s["wer"], 3), "cer": round(s["cer"], 3), "rtf": round(rtf, 4)}
    return out

for m in methods:
    print(f"{m:14}", score(m)["all"])

> **Check.** On the phone calls: Vosk small 0.316, tiny 0.206, base 0.137 and small 0.086. The real-time factors go from 0.0194 (tiny) to 0.0746 (small).

## 3. Error against speed

The next cell draws each method as a point: speed across (log scale), WER up. The best place is the bottom left.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for m in methods:
    s = score(m)["all"]
    ax.scatter(s["rtf"], s["wer"], s=60)
    ax.annotate(m, (s["rtf"], s["wer"]), textcoords="offset points", xytext=(6, 4))
ax.set_xscale("log"); ax.set_xlabel("real-time factor (log scale; smaller is faster)")
ax.set_ylabel("WER on the phone calls"); ax.set_title("Larkfield test calls: error against speed")
ax.grid(alpha=0.3); plt.show()

## 4. Predict: by slice

One number hides the slices. Larkfield's calls are in English and French, on three kinds of line, with three kinds of background.

> **Predict.** Which slice is hardest for every method: French, the mobile line, or street noise? Does `small` win on every slice?

In [ ]:
for by in ("language", "line", "noise"):
    print(f"--- WER by {by} (phone calls)")
    for m in methods:
        print(f"{m:14}", {k: v["wer"] for k, v in score(m, by=by).items()})

> **Check.** English and French: tiny 0.069 and 0.37, small 0.032 and 0.151. Look at the gaps, not only at the overall WER: a method can be good on average and bad on the slice that matters to Larkfield. Part of the French error comes from the synthetic French voices themselves.

## 5. Run Whisper tiny yourself

The recorded speeds are from one laptop. Yours are different. The next cell loads `tiny` at a fixed version and transcribes three calls.

In [ ]:
import time
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

model = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"),
                     device="cpu", compute_type="int8", cpu_threads=4)
for call_id in ['C-0225', 'C-0121', 'C-0083']:
    x16, _ = read_audio(f"data/calls/phone/{call_id}.wav", sr=16000)
    start = time.perf_counter()
    segments, _ = model.transcribe(x16, language=calls[call_id]["language"], beam_size=1, condition_on_previous_text=False)
    text = " ".join(s.text.strip() for s in segments)
    compute = time.perf_counter() - start
    recorded_rtf = recorded["whisper-tiny"][(call_id, "phone")]["seconds"] / calls[call_id]["duration"]
    print(f"{call_id}: your RTF {compute / (len(x16) / 16000):.3f}   recorded RTF {recorded_rtf:.3f}   {text[:70]}...")

> **Check.** The recorded real-time factors of these three calls are C-0225 0.015, C-0121 0.017, C-0083 0.026. On a free Colab or Kaggle CPU yours are often 2 to 5 times larger. The text should be the same as the recorded one, or very close.

## 6. Change one thing: a better line

> **Predict.** The wideband version of each call is the same speech on an HD voice line (16 kHz). How much lower is the WER?

In [ ]:
for m in methods:
    print(f"{m:14} phone WER {score(m, 'phone')['all']['wer']:.3f}   wideband WER {score(m, 'wideband')['all']['wer']:.3f}")

> **Check.** Whisper small: phone 0.086, wideband 0.078. The line itself costs accuracy. If Larkfield can get wideband recordings from its phone provider, every method improves at no compute cost.

## 7. Failure case: the mean of the per-call WERs

A common mistake is to compute a WER for each call and take the mean. A short call with three wrong words then counts as much as a long call.

In [ ]:
m = "whisper-small"
per_call = [cp.wer(cp.normalise(reference(c), "full", c["language"]),
                   cp.normalise(recorded[m][(c["call_id"], "phone")]["text"], "full", c["language"]))
            for c in test if (c["call_id"], "phone") in recorded[m]]
print(f"mean of the per-call WERs: {np.mean(per_call):.3f}   corpus WER: {score(m)['all']['wer']:.3f}")
print(f"the worst call's WER: {max(per_call):.3f}")

> **Check.** Mean of the per-call WERs 0.083, corpus WER 0.086. Here they are close, because Larkfield's test calls have similar lengths, but the worst call has a WER of 0.359. On real calls of very different lengths, the mean can move a lot with a few short calls. **Fix:** sum the errors and the reference words over all the calls, then divide (`cp.corpus_scores` does this), and report the worst calls apart.

## 8. Your turn: order numbers by size

The overall WER treats every word the same. For Larkfield, an order number with one wrong digit is a wrong order.

> **Your turn.** For each of `whisper-tiny`, `whisper-base` and `whisper-small`, count the order numbers that the recorded phone transcripts miss on the test calls. Use `cp.important_errors(call["important"], text, call["language"])` for each call, add them up with `cp.sum_important`, and put the number of missed order numbers in a dictionary `order_missed` (method → number).

In [ ]:
order_missed = {}
for m in ["whisper-tiny", "whisper-base", "whisper-small"]:
    rows = [cp.important_errors(c["important"], recorded[m][(c["call_id"], "phone")]["text"], c["language"])
            for c in test if (c["call_id"], "phone") in recorded[m]]
    total = cp.sum_important(rows)["order"]
    order_missed[m] = total["missed"]
    print(m, total)

In [ ]:
expect_hash('order_missed', order_missed, 'e9791f12d48c4dcb')

> **Check.** There are 52 order numbers in the test calls. Does the bigger model miss fewer of them, in proportion, than it misses words in general? Lesson 4 of this module helps the model with Larkfield's words and corrects order numbers after recognition.

## The comparison table

The course keeps one table with every method on the same calls. Run the next cell to see the measured rows for all the calls.

In [ ]:
table = cp.Table.load("data/comparison_table.json")
print(table.where(slice="all").show(["method", "setting", "wer", "cer", "order_missed", "rtf", "memory_mb",
                                       "usd_per_audio_hour", "privacy", "licence"]))

## Next

You compared Whisper sizes and Vosk by slice and against speed, and ran `tiny` yourself. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A hosted speech API](https://learning.nextia-ai.com/courses/speech/m04/a-hosted-speech-api/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.